# VideoMAE Fine-Tuning Pipeline

Fine-tunes **VideoMAE-Base** (Masked Autoencoder pre-trained on Kinetics-400) on the driving distraction dataset.

**Why VideoMAE over TimeSformer?**
- Pre-trained with 90-95% tube masking → much more data-efficient on small datasets
- Uses 224×224 (vs 448×448) → ~2× lower VRAM → batch=2 on P100/T4
- Same HuggingFace Trainer API

---
### Kaggle setup
1. **Add-ons → Secrets** → add `HF_TOKEN` with **write** scope.
2. Enable **Internet** and **GPU** (P100 recommended).
3. Run all cells top to bottom.

## 1. Detect Environment & Set Paths

In [ ]:
import os, sys
ON_KAGGLE = os.path.exists('/kaggle')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
if ON_KAGGLE:
    PLATFORM='kaggle'; REPO_DIR='/kaggle/working/Driving_Distraction_Detection'
    HF_CACHE_DIR='/kaggle/working/hf_cache'; OUTPUT_DIR='/kaggle/working/videomae_outputs'
elif ON_COLAB:
    PLATFORM='colab'; REPO_DIR='/content/Driving_Distraction_Detection'
    HF_CACHE_DIR='/content/hf_cache'; OUTPUT_DIR='/content/drive/MyDrive/VideoMAE_Outputs'
else:
    PLATFORM='local'; REPO_DIR=os.getcwd()
    HF_CACHE_DIR='./hf_cache'; OUTPUT_DIR='./videomae_outputs'
print(f'Platform:{PLATFORM}  Cache:{HF_CACHE_DIR}  Output:{OUTPUT_DIR}')

## 2. (Colab only) Mount Google Drive

In [ ]:
if PLATFORM=='colab':
    from google.colab import drive; drive.mount('/content/drive'); print('Mounted.')
else: print(f'Skip ({PLATFORM})')

## 3. Clone Repository

In [ ]:
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b cineca https://github.com/AnnikaUnmuessig/Driving_Distraction_Detection.git {REPO_DIR}')
else: os.system(f'git -C {REPO_DIR} pull')
os.chdir(REPO_DIR); print(f'CWD: {os.getcwd()}')

## 4. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q accelerate -U
print('Done.')

## 5. Hugging Face Authentication

In [ ]:
from huggingface_hub import login
if PLATFORM=='kaggle':
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
    print('Logged in via Kaggle Secret.')
else:
    hf_token = os.environ.get('HF_TOKEN','')
    if hf_token: login(token=hf_token, add_to_git_credential=False); print('Logged in via env.')
    else:
        from huggingface_hub import notebook_login; notebook_login()

## 6. Configuration

| Variable | Options / Description |
|---|---|
| `MODEL_VARIANT` | `'kinetics'` (general actions) or `'ssv2'` (hand/object interactions) |
| `VIDEOS_PER_CLASS` | clips to download (None = all ~13 GB) |
| `HF_REPO_ID` | `'username/repo'` to push checkpoints to HF Hub, or `''` to save locally |

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
MODEL_VARIANT    = 'kinetics'  # 'kinetics' or 'ssv2'
# Custom per-class limits (int or comma-separated name:limit string)
# Set talking_to_passenger:0 as it is not used in videomae_finetuning
VIDEOS_PER_CLASS = 'safe_driving:179,texting_right:239,phonecall_right:168,texting_left:213,phonecall_left:179,radio:147,drinking:250,reach_side:200,hair_and_makeup:266,change_gear:250,talking_to_passenger:0'
DOWNLOAD_SEED    = 42
HF_REPO_ID       = ''          # e.g. 'your_username/videomae-distraction'
USE_WANDB        = True
# ──────────────────────────────────────────────────────────────────────────────
MODEL_ID_MAP = {
    'kinetics': 'MCG-NJU/videomae-base-finetuned-kinetics',
    'ssv2':     'MCG-NJU/videomae-base-finetuned-ssv2',
}
MODEL_HF_ID = MODEL_ID_MAP.get(MODEL_VARIANT, MODEL_ID_MAP['kinetics'])
print(f'Model  : {MODEL_VARIANT} → {MODEL_HF_ID}')
print(f'Videos : {VIDEOS_PER_CLASS}')
print(f'HF Hub : {HF_REPO_ID or "(disabled)"}')

## 7. Download Model & Dataset

In [ ]:
# Il modello VideoMAE viene scaricato automaticamente da HF Hub durante il training.
# Qui scaricamo solo il dataset video.
cmd = f'python download_assets.py --output_dir {HF_CACHE_DIR} --seed {DOWNLOAD_SEED}'
if VIDEOS_PER_CLASS: cmd += f' --videos_per_class {VIDEOS_PER_CLASS}'
print(f'Running: {cmd}\n')
!{cmd}

## 8. Start Training

In [ ]:
# MODEL_PATH = HF Hub model ID → scaricato/cachato automaticamente da HuggingFace
os.environ['MODEL_PATH']       = MODEL_HF_ID
os.environ['DATASET_PATH']     = os.path.join(HF_CACHE_DIR, 'distraction_dataset')
os.environ['OUTPUT_DIR']       = OUTPUT_DIR
os.environ['HF_REPO_ID']       = HF_REPO_ID
os.environ['TRAIN_BATCH_SIZE'] = '2'   # P100 gestisce batch=2 a 224x224
os.environ['EVAL_BATCH_SIZE']  = '2'
os.environ['GRAD_ACCUM_STEPS'] = '8'   # effective batch = 16
if VIDEOS_PER_CLASS: os.environ['LIMIT_CAP'] = str(VIDEOS_PER_CLASS)

if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        wkey = UserSecretsClient().get_secret('WANDB_API_KEY')
        if wkey:
            os.environ['WANDB_API_KEY'] = wkey
            print('Loaded WANDB_API_KEY from Kaggle Secrets.')
    except Exception:
        pass
    import wandb
    try:
        wandb.login(key=os.environ.get('WANDB_API_KEY'))
        print('WandB login successful.')
    except Exception as e:
        print(f'WandB login prompt: {e}')
else:
    os.environ['WANDB_MODE']='disabled'; os.environ['WANDB_DISABLED']='true'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'MODEL_PATH   = {os.environ["MODEL_PATH"]}')
print(f'DATASET_PATH = {os.environ["DATASET_PATH"]}')
print(f'OUTPUT_DIR   = {os.environ["OUTPUT_DIR"]}')
print(f'HF_REPO_ID   = {os.environ["HF_REPO_ID"] or "(disabled)"}')
print()
!python videomae_finetuning.py

## 9. Inspect Outputs

In [ ]:
if os.path.exists(OUTPUT_DIR):
    for root, dirs, files in os.walk(OUTPUT_DIR):
        lvl = root.replace(OUTPUT_DIR,'').count(os.sep)
        pad = '  '*lvl
        print(f'{pad}{os.path.basename(root)}/')
        for f in files:
            mb = os.path.getsize(os.path.join(root,f))/1024**2
            print(f'{pad}  {f}  ({mb:.1f} MB)')
else: print('Output dir not found.')